# Day 8 v3 — Silver Layer: All 17 Entities (Production + Data Quality)
**Bronze Volume JSON → Silver Delta Tables**

**Source:** `/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/api/<entity>/`
**Sink:** `/Volumes/dbw_ev_intelligence_dev/default/silver-volume/api/<entity>/` (Delta)

**What v3 adds over v2 (for loop version):**

| Cell | Step | What it does |
|---|---|---|
| Cell 7 | Read + Explode | Read Bronze JSON, explode `data[]` into flat rows |
| Cell 8 | Trim whitespace | Strip spaces from all string columns |
| Cell 9 | Null primary key | Drop + quarantine rows with missing natural key |
| Cell 10 | Null CDC field | Drop + quarantine rows with missing `updated_at` |
| Cell 11 | Cast types + corrupt check | Cast columns, quarantine rows where numeric cast fails |
| Cell 12 | Negative values | Quarantine rows with negative amounts or prices |
| Cell 13 | Audit columns + Dedup | Add Silver metadata columns, keep latest record per key |
| Cell 14 | Write to Silver | Overwrite (full) or Delta MERGE upsert (incremental) |
| Cell 15 | Orchestrator | `transform_entity()` calls all steps in sequence for one entity |
| Cell 16 | Run all 17 entities | Loop over ENTITY_REGISTRY, collect results |
| Cell 17 | Run summary | Per-entity breakdown: bronze / rejected / silver row counts |
| Cell 18 | Verify | Spot-check Silver + quarantine tables |

In [ ]:
# Cell 1 — Imports
from pyspark.sql import functions as F
from pyspark.sql.types import MapType, StringType
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime, timezone

print("Imports OK")

In [ ]:
# Cell 2 — Widgets
dbutils.widgets.removeAll()
dbutils.widgets.text("load_type", "incremental", "Load Type (full / incremental)")
dbutils.widgets.text("ingestion_date", "", "Ingestion Date (YYYY-MM-DD, blank = all)")

LOAD_TYPE      = dbutils.widgets.get("load_type").strip().lower()
INGESTION_DATE = dbutils.widgets.get("ingestion_date").strip()

if LOAD_TYPE not in ("full", "incremental"):
    raise ValueError(f"load_type must be 'full' or 'incremental', got: {LOAD_TYPE!r}")

print(f"load_type      : {LOAD_TYPE}")
print(f"ingestion_date : {INGESTION_DATE or '(all partitions)'}")

In [ ]:
# Cell 3 — Constants
BRONZE_VOLUME = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume"
SILVER_VOLUME = "/Volumes/dbw_ev_intelligence_dev/default/silver-volume"

BRONZE_API_BASE = f"{BRONZE_VOLUME}/api"
SILVER_API_BASE = f"{SILVER_VOLUME}/api"
QUARANTINE_BASE = f"{SILVER_VOLUME}/quarantine"   # rejected rows land here

PIPELINE_NAME = "pl_silver_api_transformation_v3"
RUN_TS        = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

print(f"BRONZE_API_BASE : {BRONZE_API_BASE}")
print(f"SILVER_API_BASE : {SILVER_API_BASE}")
print(f"QUARANTINE_BASE : {QUARANTINE_BASE}")
print(f"RUN_TS          : {RUN_TS}")

In [ ]:
# Cell 4 — Entity schema registry
# natural_key : unique ID column used for deduplication and Delta MERGE
# cdc_field   : timestamp column that determines which version of a record is latest
# cast        : column -> spark type mapping for explicit type casting

ENTITY_REGISTRY = {
    "payments": {
        "natural_key": "payment_id", "cdc_field": "updated_at",
        "cast": {
            "payment_id": "string", "session_id": "string", "customer_id": "string",
            "amount_aud": "decimal(10,2)", "currency": "string", "status": "string",
            "payment_method": "string", "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    "sessions": {
        "natural_key": "session_id", "cdc_field": "updated_at",
        "cast": {
            "session_id": "string", "vehicle_id": "string", "charger_id": "string",
            "customer_id": "string", "station_id": "string",
            "start_time": "timestamp", "end_time": "timestamp",
            "energy_kwh": "decimal(10,4)", "duration_minutes": "integer",
            "status": "string", "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    "customers": {
        "natural_key": "customer_id", "cdc_field": "updated_at",
        "cast": {
            "customer_id": "string", "first_name": "string", "last_name": "string",
            "email": "string", "phone": "string", "city": "string",
            "state": "string", "country": "string",
            "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    "fleet": {
        "natural_key": "vehicle_id", "cdc_field": "updated_at",
        "cast": {
            "vehicle_id": "string", "make": "string", "model": "string", "year": "integer",
            "battery_capacity": "decimal(8,2)", "range_km": "decimal(8,2)",
            "status": "string", "partner_id": "string",
            "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    "chargers": {
        "natural_key": "charger_id", "cdc_field": "updated_at",
        "cast": {
            "charger_id": "string", "station_id": "string", "charger_type": "string",
            "power_kw": "decimal(8,2)", "status": "string", "connector_type": "string",
            "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    "vehicles": {
        "natural_key": "vehicle_id", "cdc_field": "updated_at",
        "cast": {
            "vehicle_id": "string", "customer_id": "string", "make": "string",
            "model": "string", "year": "integer",
            "battery_capacity": "decimal(8,2)", "range_km": "decimal(8,2)",
            "license_plate": "string", "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    "stations": {
        "natural_key": "station_id", "cdc_field": "updated_at",
        "cast": {
            "station_id": "string", "name": "string", "city": "string",
            "state": "string", "country": "string",
            "latitude": "decimal(10,7)", "longitude": "decimal(10,7)",
            "total_chargers": "integer", "status": "string",
            "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    "complaints": {
        "natural_key": "complaint_id", "cdc_field": "updated_at",
        "cast": {
            "complaint_id": "string", "customer_id": "string", "session_id": "string",
            "category": "string", "description": "string", "status": "string",
            "priority": "string", "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    "maintenance_events": {
        "natural_key": "event_id", "cdc_field": "updated_at",
        "cast": {
            "event_id": "string", "charger_id": "string", "station_id": "string",
            "event_type": "string", "description": "string", "technician_id": "string",
            "status": "string", "scheduled_date": "date", "completed_date": "date",
            "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    "energy_prices": {
        "natural_key": "price_id", "cdc_field": "updated_at",
        "cast": {
            "price_id": "string", "region": "string",
            "price_per_kwh": "decimal(8,4)", "currency": "string",
            "effective_from": "timestamp", "effective_to": "timestamp",
            "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    "tariffs": {
        "natural_key": "tariff_id", "cdc_field": "updated_at",
        "cast": {
            "tariff_id": "string", "name": "string",
            "price_per_kwh": "decimal(8,4)", "price_per_min": "decimal(8,4)",
            "currency": "string", "valid_from": "timestamp", "valid_to": "timestamp",
            "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    "charge_cards": {
        "natural_key": "card_id", "cdc_field": "updated_at",
        "cast": {
            "card_id": "string", "customer_id": "string", "card_number": "string",
            "status": "string", "issued_at": "timestamp", "expires_at": "timestamp",
            "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    "employees": {
        "natural_key": "employee_id", "cdc_field": "updated_at",
        "cast": {
            "employee_id": "string", "first_name": "string", "last_name": "string",
            "email": "string", "role": "string", "department": "string",
            "station_id": "string", "hire_date": "date",
            "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    "partners": {
        "natural_key": "partner_id", "cdc_field": "updated_at",
        "cast": {
            "partner_id": "string", "name": "string", "country": "string",
            "contact_email": "string", "status": "string",
            "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    "cities": {
        "natural_key": "city_id", "cdc_field": "updated_at",
        "cast": {
            "city_id": "string", "name": "string", "state_code": "string",
            "country": "string", "latitude": "decimal(10,7)", "longitude": "decimal(10,7)",
            "population": "long", "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    "states": {
        "natural_key": "state_code", "cdc_field": "updated_at",
        "cast": {
            "state_code": "string", "name": "string", "country": "string",
            "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    "weather": {
        "natural_key": "city_id", "cdc_field": "updated_at",
        "cast": {
            "city_id": "string", "temperature_c": "decimal(5,2)",
            "humidity_pct": "decimal(5,2)", "wind_speed_kmh": "decimal(6,2)",
            "condition": "string", "recorded_at": "timestamp",
            "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
}

print(f"Entity registry loaded: {len(ENTITY_REGISTRY)} entities")

In [ ]:
# Cell 5 — Shared helper functions

def get_bronze_paths(entity_name, ingestion_date):
    """Return list of Bronze JSON file paths for the entity.
    If ingestion_date given -> one partition. Else -> all partitions (full load)."""
    base = f"{BRONZE_API_BASE}/{entity_name}"
    try:
        dbutils.fs.ls(base)
    except Exception:
        return []
    if ingestion_date:
        try:
            files = dbutils.fs.ls(f"{base}/ingestion_date={ingestion_date}")
            return [f.path for f in files if f.path.endswith(".json")]
        except Exception:
            return []
    else:
        all_files = []
        for p in dbutils.fs.ls(base):
            try:
                all_files.extend([f.path for f in dbutils.fs.ls(p.path) if f.path.endswith(".json")])
            except Exception:
                pass
        return all_files


def folder_exists_dbfs(path):
    """Return True if path exists in DBFS/Volume."""
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


def write_quarantine(df, entity_name, reason):
    """Append rejected rows to the quarantine Delta table with reject_reason tag."""
    # Use limit(1).count() instead of rdd.isEmpty() — rdd is not available on Serverless
    if df.limit(1).count() == 0:
        return
    (
        df.withColumn("reject_reason", F.lit(reason))
          .withColumn("reject_ts",     F.lit(RUN_TS).cast("timestamp"))
          .write.format("delta")
          .mode("append")
          .option("mergeSchema", "true")
          .save(f"{QUARANTINE_BASE}/{entity_name}")
    )


print("Helpers defined: get_bronze_paths, folder_exists_dbfs, write_quarantine")

In [ ]:
# Cell 6 — Negative value check config
# Columns that must never hold negative values in this domain.
# Rows that violate this go to quarantine instead of Silver.

NEGATIVE_CHECK_COLS = {
    "payments"     : ["amount_aud"],
    "sessions"     : ["energy_kwh", "duration_minutes"],
    "fleet"        : ["battery_capacity", "range_km"],
    "chargers"     : ["power_kw"],
    "vehicles"     : ["battery_capacity", "range_km"],
    "stations"     : ["total_chargers"],
    "energy_prices": ["price_per_kwh"],
    "tariffs"      : ["price_per_kwh", "price_per_min"],
    "cities"       : ["population"],
    "weather"      : ["humidity_pct"],
}

print("NEGATIVE_CHECK_COLS defined")

In [ ]:
# Cell 7 — Step 1: Read Bronze JSON + Explode data[] array
# Each Bronze file: { "data": [...records...], "pagination": {...} }
# explode() turns the array into one row per record.
#
# array<struct> path -> select("r.*") works fine
# array<string> path -> from_json produces map<string,string>
#                       map does NOT support "r.*" — extract keys by name from cast_map

def step_read_and_explode(paths, cast_map):
    raw_df = spark.read.option("multiline", "true").json(paths)

    if "data" not in raw_df.columns:
        raise ValueError("Column 'data' not found in Bronze JSON — schema mismatch")

    exploded_df = raw_df.select(F.explode(F.col("data")).alias("r"))
    r_dtype = dict(exploded_df.dtypes)["r"]

    if r_dtype == "string":
        # Parse JSON string into map<string,string>, then extract each column by name
        parsed_df = exploded_df.select(
            F.from_json(F.col("r"), MapType(StringType(), StringType())).alias("r")
        )
        col_exprs = [F.col("r")[k].alias(k) for k in cast_map.keys()]
        flat_df   = parsed_df.select(col_exprs)
    else:
        # array<struct> — expand struct fields directly
        flat_df = exploded_df.select("r.*")

    return flat_df

print("step_read_and_explode() defined")

In [ ]:
# Cell 8 — Step 2: Trim whitespace from all string columns
# Bronze strings can arrive as "  pending  " or "completed " from the API.
# Trailing/leading spaces cause silent mismatches in status filters and joins.

def step_trim_strings(df):
    string_cols = [c for c, t in df.dtypes if t == "string"]
    for col in string_cols:
        df = df.withColumn(col, F.trim(F.col(col)))
    return df

print("step_trim_strings() defined")

In [ ]:
# Cell 9 — Step 3: Drop rows with NULL or empty primary key
# A record with no natural key cannot be identified, deduplicated, or merged.
# These rows go to quarantine — they are not silently dropped.

def step_filter_null_pk(df, entity_name, natural_key):
    null_pk_df = df.filter(F.col(natural_key).isNull() | (F.col(natural_key) == ""))
    count = null_pk_df.count()
    if count > 0:
        write_quarantine(null_pk_df, entity_name, f"null_primary_key:{natural_key}")
    clean_df = df.filter(F.col(natural_key).isNotNull() & (F.col(natural_key) != ""))
    return clean_df, count

print("step_filter_null_pk() defined")

In [ ]:
# Cell 10 — Step 4: Drop rows with NULL or empty CDC timestamp
# updated_at is used to pick the latest version during deduplication.
# Without it we cannot know if a record is old or new — quarantine it.

def step_filter_null_cdc(df, entity_name, cdc_field):
    null_cdc_df = df.filter(F.col(cdc_field).isNull() | (F.col(cdc_field) == ""))
    count = null_cdc_df.count()
    if count > 0:
        write_quarantine(null_cdc_df, entity_name, f"null_cdc_field:{cdc_field}")
    clean_df = df.filter(F.col(cdc_field).isNotNull() & (F.col(cdc_field) != ""))
    return clean_df, count

print("step_filter_null_cdc() defined")

In [ ]:
# Cell 11 — Step 5: Cast columns to correct types + detect corrupt rows
#
# A corrupt row = numeric column that had a value in Bronze but became NULL after cast
# (e.g. "abc" cast to decimal returns NULL — that is corrupt data).
#
# We must NOT flag rows where the column was already NULL in Bronze —
# those are legitimately missing values, not corrupt.
# Pre-cast sentinel: record whether each numeric column was non-null before casting.

def step_cast_and_check_corrupt(flat_df, entity_name, cast_map):
    numeric_types = ("decimal", "integer", "long", "double", "float")

    numeric_cols = [
        c for c, t in cast_map.items()
        if any(t.startswith(nt) for nt in numeric_types)
        and c in flat_df.columns
    ]

    # Add a sentinel column per numeric col: True if Bronze value was non-null/non-empty
    pre_cast_flags = {}
    for c in numeric_cols:
        flag_col = f"_pre_{c}"
        flat_df = flat_df.withColumn(
            flag_col,
            F.col(c).isNotNull() & (F.col(c).cast("string") != "")
        )
        pre_cast_flags[c] = flag_col

    # Build cast expressions
    cast_exprs = []
    for col_name, col_type in cast_map.items():
        if col_name in flat_df.columns:
            cast_exprs.append(F.col(col_name).cast(col_type).alias(col_name))
        else:
            cast_exprs.append(F.lit(None).cast(col_type).alias(col_name))

    registry_cols = set(cast_map.keys())
    flag_cols     = list(pre_cast_flags.values())
    extra_cols    = [F.col(c) for c in flat_df.columns
                     if c not in registry_cols and c not in flag_cols]

    typed_df = flat_df.select(cast_exprs + extra_cols + [F.col(f) for f in flag_cols])

    # Detect corrupt: pre-cast non-null AND post-cast null => unparseable value
    corrupt_count = 0
    if numeric_cols:
        corrupt_condition = F.lit(False)
        for c, flag in pre_cast_flags.items():
            corrupt_condition = corrupt_condition | (F.col(flag) & F.col(c).isNull())

        corrupt_df    = typed_df.filter(corrupt_condition)
        corrupt_count = corrupt_df.drop(*flag_cols).count()
        if corrupt_count > 0:
            write_quarantine(corrupt_df.drop(*flag_cols), entity_name, "corrupt_cast_to_null")
        typed_df = typed_df.filter(~corrupt_condition)

    typed_df = typed_df.drop(*flag_cols)

    return typed_df, corrupt_count

print("step_cast_and_check_corrupt() defined")

In [ ]:
# Cell 12 — Step 6: Quarantine rows with negative numeric values
# Domain rule: amounts, energy, capacity, prices, population cannot be negative.
# Negative values indicate bad source data — quarantine them, do not write to Silver.

def step_filter_negative_values(df, entity_name):
    neg_cols = [c for c in NEGATIVE_CHECK_COLS.get(entity_name, []) if c in df.columns]

    if not neg_cols:
        return df, 0

    negative_condition = F.lit(False)
    for c in neg_cols:
        negative_condition = negative_condition | (F.col(c) < 0)

    negative_df = df.filter(negative_condition)
    count       = negative_df.count()
    if count > 0:
        write_quarantine(negative_df, entity_name, "negative_numeric_value")
    clean_df = df.filter(~negative_condition)
    return clean_df, count

print("step_filter_negative_values() defined")

In [ ]:
# Cell 13 — Step 7: Add Silver audit columns + Deduplicate on natural key
#
# Audit columns: every Silver row gets silver_ingested_at, silver_load_type,
# silver_pipeline so we always know when and how a record arrived in Silver.
#
# Deduplication: Bronze can have the same natural_key across multiple JSON pages
# or ingestion dates (same record updated multiple times).
# Window function keeps only the row with the most recent updated_at per key.

def step_add_audit_and_dedup(df, natural_key, cdc_field, load_type):
    # Add audit columns
    df = (
        df
        .withColumn("silver_ingested_at", F.lit(RUN_TS).cast("timestamp"))
        .withColumn("silver_load_type",   F.lit(load_type))
        .withColumn("silver_pipeline",    F.lit(PIPELINE_NAME))
    )

    # Deduplicate: keep latest updated_at per natural_key
    before_count = df.count()
    window = Window.partitionBy(natural_key).orderBy(F.col(cdc_field).desc())
    deduped_df = (
        df
        .withColumn("_row_num", F.row_number().over(window))
        .filter(F.col("_row_num") == 1)
        .drop("_row_num")
    )
    duplicate_count = before_count - deduped_df.count()

    return deduped_df, duplicate_count

print("step_add_audit_and_dedup() defined")

In [ ]:
# Cell 14 — Step 8: Write to Silver Delta table
#
# Full load  -> overwrite the entire Silver table from scratch.
# Incremental -> Delta MERGE (upsert):
#     - matched key   -> UPDATE all columns to latest values
#     - new key       -> INSERT as new row
#     - key only in Silver (not in today's batch) -> left unchanged (no delete)

def step_write_silver(df, entity_name, natural_key, silver_path, load_type):
    writer = df.write.format("delta").option("mergeSchema", "true")

    if load_type == "full" or not folder_exists_dbfs(silver_path):
        writer.mode("overwrite").save(silver_path)
    else:
        if DeltaTable.isDeltaTable(spark, silver_path):
            delta_table = DeltaTable.forPath(spark, silver_path)
            (
                delta_table.alias("target")
                .merge(
                    df.alias("source"),
                    f"target.{natural_key} = source.{natural_key}"
                )
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )
        else:
            # Silver path exists but is not Delta yet — create it
            writer.mode("overwrite").save(silver_path)

print("step_write_silver() defined")

In [ ]:
# Cell 15 — Orchestrator: transform_entity()
# Calls all step functions in sequence for a single entity.
# Returns a result dict with row counts for every quality check.

def transform_entity(entity_name, config, load_type, ingestion_date):
    result = {
        "entity_name"        : entity_name,
        "status"             : "failed",
        "bronze_rows"        : 0,
        "null_pk_dropped"    : 0,
        "null_cdc_dropped"   : 0,
        "corrupt_quarantine" : 0,
        "negative_quarantine": 0,
        "duplicate_dropped"  : 0,
        "silver_rows"        : 0,
        "error"              : None,
    }
    try:
        natural_key = config["natural_key"]
        cdc_field   = config["cdc_field"]
        cast_map    = config["cast"]
        silver_path = f"{SILVER_API_BASE}/{entity_name}"

        paths = get_bronze_paths(entity_name, ingestion_date)
        if not paths:
            result["status"] = "skipped"
            result["error"]  = "No Bronze JSON files found"
            return result

        # Step 1 — Read + Explode (cast_map passed for array<string> column extraction)
        flat_df = step_read_and_explode(paths, cast_map)
        result["bronze_rows"] = flat_df.count()

        # Step 2 — Trim whitespace
        flat_df = step_trim_strings(flat_df)

        # Step 3 — Drop null primary key
        flat_df, result["null_pk_dropped"] = step_filter_null_pk(flat_df, entity_name, natural_key)

        # Step 4 — Drop null CDC field
        flat_df, result["null_cdc_dropped"] = step_filter_null_cdc(flat_df, entity_name, cdc_field)

        # Step 5 — Cast types + detect corrupt rows
        typed_df, result["corrupt_quarantine"] = step_cast_and_check_corrupt(flat_df, entity_name, cast_map)

        # Step 6 — Quarantine negative numeric values
        typed_df, result["negative_quarantine"] = step_filter_negative_values(typed_df, entity_name)

        # Step 7 — Add audit columns + deduplicate
        deduped_df, result["duplicate_dropped"] = step_add_audit_and_dedup(typed_df, natural_key, cdc_field, load_type)

        # Step 8 — Write to Silver
        step_write_silver(deduped_df, entity_name, natural_key, silver_path, load_type)

        result["silver_rows"] = deduped_df.count()
        result["status"]      = "succeeded"

    except Exception as e:
        result["error"] = str(e)

    return result

print("transform_entity() orchestrator defined")

In [ ]:
# Cell 16 — Run transformation for all 17 entities
# Sequential loop — PySpark parallelises reads/writes internally per entity.

print(f"Starting Silver transformation — load_type={LOAD_TYPE}")
print("-" * 60)

results = []
for entity_name, config in ENTITY_REGISTRY.items():
    date_filter = INGESTION_DATE if LOAD_TYPE == "incremental" else ""
    print(f"  Processing: {entity_name} ...", end=" ")
    r = transform_entity(entity_name, config, LOAD_TYPE, date_filter)
    results.append(r)
    if r["status"] == "succeeded":
        print(f"OK  (bronze={r['bronze_rows']} -> silver={r['silver_rows']})")
    elif r["status"] == "skipped":
        print(f"SKIP  ({r['error']})")
    else:
        print(f"FAILED  ({r['error']})")

print("-" * 60)
print("All entities processed")

In [ ]:
# Cell 17 — Run summary with data quality breakdown per entity

succeeded = [r for r in results if r["status"] == "succeeded"]
skipped   = [r for r in results if r["status"] == "skipped"]
failed    = [r for r in results if r["status"] == "failed"]

print("=" * 95)
print("SILVER API TRANSFORMATION v3 — RUN SUMMARY")
print("=" * 95)
print(f"  load_type      : {LOAD_TYPE}")
print(f"  ingestion_date : {INGESTION_DATE or '(all partitions)'}")
print(f"  run_timestamp  : {RUN_TS}")
print(f"  entities total : {len(results)}  |  succeeded: {len(succeeded)}  |  skipped: {len(skipped)}  |  failed: {len(failed)}")
print()
print(f"  {'ENTITY':<25} {'STATUS':<10} {'BRONZE':>7} {'NULL_PK':>8} {'NULL_CDC':>9} {'CORRUPT':>8} {'NEGATIVE':>9} {'DUPS':>6} {'SILVER':>7}")
print(f"  {'-'*25} {'-'*10} {'-'*7} {'-'*8} {'-'*9} {'-'*8} {'-'*9} {'-'*6} {'-'*7}")

for r in results:
    tag = "[OK]  " if r["status"] == "succeeded" else ("[SKIP]" if r["status"] == "skipped" else "[FAIL]")
    print(
        f"  {tag} {r['entity_name']:<25} {r['status']:<10}"
        f" {r['bronze_rows']:>7}"
        f" {r['null_pk_dropped']:>8}"
        f" {r['null_cdc_dropped']:>9}"
        f" {r['corrupt_quarantine']:>8}"
        f" {r['negative_quarantine']:>9}"
        f" {r['duplicate_dropped']:>6}"
        f" {r['silver_rows']:>7}"
    )
    if r["error"]:
        print(f"         Error: {r['error']}")

print()
total_quarantined = sum(
    r["null_pk_dropped"] + r["null_cdc_dropped"] + r["corrupt_quarantine"] + r["negative_quarantine"]
    for r in results
)
print(f"  Total rows quarantined : {total_quarantined}  (saved to {QUARANTINE_BASE}/<entity>/)")
print(f"  Total rows in Silver   : {sum(r['silver_rows'] for r in results)}")
print()

if failed:
    raise Exception(f"{len(failed)} entity(ies) failed — check output above.")
else:
    print(f"Silver transformation complete. {len(succeeded)} entities written successfully.")

In [ ]:
# Cell 18 — Verify Silver + Quarantine (spot-check 3 entities)

print("SILVER VERIFICATION — spot check")
print("-" * 50)
for entity in ["payments", "sessions", "customers"]:
    silver_path = f"{SILVER_API_BASE}/{entity}"
    if not folder_exists_dbfs(silver_path):
        print(f"  {entity}: NOT FOUND at {silver_path}")
        continue
    df = spark.read.format("delta").load(silver_path)
    print(f"  {entity:<25} rows={df.count():>6}  cols={len(df.columns)}")
    if entity == "payments":
        print()
        df.printSchema()
        df.show(3, truncate=True)

print()
print("QUARANTINE CHECK — rows rejected by data quality rules")
print("-" * 50)
for entity in ["payments", "sessions", "customers"]:
    q_path = f"{QUARANTINE_BASE}/{entity}"
    if not folder_exists_dbfs(q_path):
        print(f"  {entity:<25} no quarantine rows (data is clean)")
        continue
    q_df = spark.read.format("delta").load(q_path)
    print(f"  {entity:<25} quarantine rows={q_df.count()}")
    q_df.groupBy("reject_reason").count().orderBy("count", ascending=False).show(truncate=False)